# Dolomite in NaCl — Visual MINTEQ-style speciation (fixed pCO₂, pH from mass balance)

This notebook reproduces the Visual MINTEQ speciation for the dolomite titration solutions. It
follows the same recipe:

* the carbonate system is **open** — CO₂(aq) is held at a fixed activity (fixed pCO₂),
* **pH is calculated from the mass balance** (MINTEQ option 1), not read from the electrode,
* the free-ion **H⁺ starts from a neutral guess (10⁻⁷)** and is solved from the proton condition,
* Davies activity coefficients and the TOUGHREACT / EQ3-6 (Plummer–Busenberg) constants are used,
* no surface complexation model.

Every equation is written out below: the reactions, the law of mass action for each species, the
element mass balances, and the proton condition that fixes pH. The net surface charge
(Pokrovsky Eqn 1) is computed at the calculated pH at the end.

## 1. Reactions and equilibrium constants (25 °C, log₁₀K)

Written as dissociations (complex on the left), from the TOUGHREACT / EQ3-6 database.

**Water and carbonate**

| reaction | log K |
|----------|-------|
| H₂O = H⁺ + OH⁻ | −13.995 |
| CO₂(aq) + H₂O = H⁺ + HCO₃⁻ | −6.345 |
| HCO₃⁻ = H⁺ + CO₃²⁻ | −10.329 |
| CO₂(g) = CO₂(aq) → {CO₂(aq)} fixed by pCO₂ | (fixed) |

**Calcium / magnesium / sodium complexes**

| reaction | log K | reaction | log K |
|----------|-------|----------|-------|
| CaCl⁺ = Ca²⁺ + Cl⁻ | 0.696 | MgCl⁺ = Mg²⁺ + Cl⁻ | 0.135 |
| CaCO₃(aq) + H⁺ = Ca²⁺ + HCO₃⁻ | 7.002 | MgCO₃(aq) + H⁺ = Mg²⁺ + HCO₃⁻ | 7.350 |
| CaHCO₃⁺ = Ca²⁺ + HCO₃⁻ | −1.047 | MgHCO₃⁺ = Mg²⁺ + HCO₃⁻ | −1.036 |
| CaOH⁺ + H⁺ = Ca²⁺ + H₂O | 12.850 | MgOH⁺ + H⁺ = Mg²⁺ + H₂O | 11.785 |
| NaCl(aq) = Na⁺ + Cl⁻ | 0.777 | NaCO₃⁻ + H⁺ = Na⁺ + HCO₃⁻ | 9.815 |
| NaHCO₃(aq) = Na⁺ + HCO₃⁻ | −0.154 | NaOH(aq) + H⁺ = Na⁺ + H₂O | 14.180 |


In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import brentq

# log10 K of each dissociation (25 C, TOUGHREACT / EQ3-6)
lk = dict(co2=-6.345, co3=10.329, oh=13.995,
          cacl=0.696, caco3=7.002, cahco3=-1.047, caoh=12.850,
          mgcl=0.135, mgco3=7.350, mghco3=-1.036, mgoh=11.785,
          nacl=0.777, naco3=9.815, nahco3=-0.154, naoh=14.180)

aCO2 = 10**-4.90      # fixed {CO2(aq)} activity  ==  fixed pCO2  (MINTEQ log activity = -4.9)
A_DAVIES = 0.509
MM = dict(Ca=40.078, Mg=24.305, Na=22.99, Cl=35.45, CO3=60.008, HCO3=61.016)

# surface area of dolomite per litre of reactor C  (Pokrovsky Eqn 1)
St = 0.84 * 30.0      # 0.84 m2/g * 30 g/L = 25.2 m2/L

## 2. Activity coefficients (Davies)

$$\log\gamma_z=-A\,z^2\!\left(\frac{\sqrt I}{1+\sqrt I}-0.3\,I\right),\qquad
\log\gamma_0=0.1\,I\ \text{(neutral species)}$$

Activities are $\{i\}=\gamma_{z_i}\,[i]$; for neutral species $\gamma_0$ is the Setchenow term MINTEQ
applies (this is why {H₂CO₃*} activity ≈ 1.26 × its concentration).

In [2]:
def gammas(I):
    if I <= 0:
        return 1.0, 1.0, 1.0
    f = -A_DAVIES * (np.sqrt(I)/(1+np.sqrt(I)) - 0.3*I)
    return 10**f, 10**(4*f), 10**(0.1*I)     # gamma1 (z=1), gamma2 (z=2), gamma0 (neutral)

## 3. Law of mass action and the element mass balances

With `aH = {H⁺}` and the fixed `aCO2`, the carbonate activities are

$$\{HCO_3^-\}=10^{-6.345}\,\frac{\{CO_2\}}{\{H^+\}},\quad
\{CO_3^{2-}\}=\frac{\{HCO_3^-\}}{10^{10.329}\{H^+\}},\quad
\{OH^-\}=\frac{10^{-13.995}}{\{H^+\}}$$

Each complex activity is its mass-action law, e.g. $\{CaHCO_3^+\}=10^{1.047}\{Ca^{2+}\}\{HCO_3^-\}$,
$\{CaCl^+\}=\{Ca^{2+}\}\{Cl^-\}/10^{0.696}$, $\{CaOH^+\}=\{Ca^{2+}\}/(10^{12.850}\{H^+\})$.

The four **mass balances** (free ion = total − everything it is bound into; no surface terms) close the
system for the free-ion activities `aCa, aMg, aNa, aCl`:

```
Ca_T = [Ca²⁺] + [CaCl⁺] + [CaCO₃] + [CaHCO₃⁺] + [CaOH⁺]
Mg_T = [Mg²⁺] + [MgCl⁺] + [MgCO₃] + [MgHCO₃⁺] + [MgOH⁺]
Na_T = [Na⁺]  + [NaCl]   + [NaCO₃⁻] + [NaHCO₃] + [NaOH]
Cl_T = [Cl⁻]  + [CaCl⁺] + [MgCl⁺] + [NaCl]
```

Because every metal species is proportional to its free-ion activity, each metal total gives
`aMe = Me_T / d_Me` with `d_Me` the bracketed sum; `aCl` follows from the Cl balance. The two are
iterated (and the ionic strength around them) to convergence.

In [3]:
def speciate(pH, CaT, MgT, NaT, ClT):
    """Full speciation at a fixed pH and fixed pCO2 (Davies activities)."""
    aH = 10**(-pH)
    I  = 0.5*(NaT + ClT + 4*CaT + 4*MgT)
    aCa = aMg = aNa = aCl = 0.0
    for _ in range(200):
        g1, g2, g0 = gammas(I)
        aHCO3 = 10**lk['co2']*aCO2/aH          # {HCO3-}
        aCO3  = aHCO3/(10**lk['co3']*aH)       # {CO3^2-}
        aOH   = 10**(-lk['oh'])/aH             # {OH-}
        cH, cOH, cHCO3, cCO3, cCO2 = aH/g1, aOH/g1, aHCO3/g1, aCO3/g2, aCO2/g0
        if aCl == 0:
            aCa, aMg, aNa, aCl = g2*CaT, g2*MgT, g1*NaT, g1*ClT
        for _ in range(100):
            dCa = (1/g2 + aCl/(g1*10**lk['cacl']) + aHCO3/(g0*aH*10**lk['caco3'])
                   + aHCO3*10**(-lk['cahco3'])/g1 + 1/(g1*aH*10**lk['caoh']))
            dMg = (1/g2 + aCl/(g1*10**lk['mgcl']) + aHCO3/(g0*aH*10**lk['mgco3'])
                   + aHCO3*10**(-lk['mghco3'])/g1 + 1/(g1*aH*10**lk['mgoh']))
            dNa = (1/g1 + aHCO3/(g1*aH*10**lk['naco3']) + aHCO3*10**(-lk['nahco3'])/g0
                   + aCl/(g0*10**lk['nacl']) + 1/(g0*aH*10**lk['naoh']))
            aCa, aMg, aNa = CaT/dCa, MgT/dMg, NaT/dNa
            dCl = (1/g1 + aCa/(g1*10**lk['cacl']) + aMg/(g1*10**lk['mgcl']) + aNa/(g0*10**lk['nacl']))
            aCl_new = ClT/dCl
            if abs(aCl_new-aCl) < 1e-12: aCl = aCl_new; break
            aCl = aCl_new
        # complex concentrations
        cCaCl  = aCa*aCl/(g1*10**lk['cacl']);   cCaCO3 = aCa*aHCO3/(g0*aH*10**lk['caco3'])
        cCaHCO3= aCa*aHCO3*10**(-lk['cahco3'])/g1; cCaOH = aCa/(g1*aH*10**lk['caoh'])
        cMgCl  = aMg*aCl/(g1*10**lk['mgcl']);   cMgCO3 = aMg*aHCO3/(g0*aH*10**lk['mgco3'])
        cMgHCO3= aMg*aHCO3*10**(-lk['mghco3'])/g1; cMgOH = aMg/(g1*aH*10**lk['mgoh'])
        cNaCl  = aNa*aCl/(g0*10**lk['nacl']);   cNaCO3 = aNa*aHCO3/(g1*aH*10**lk['naco3'])
        cNaHCO3= aNa*aHCO3*10**(-lk['nahco3'])/g0; cNaOH = aNa/(g0*aH*10**lk['naoh'])
        cCa, cMg, cNa, cCl = aCa/g2, aMg/g2, aNa/g1, aCl/g1
        Inew = 0.5*(cNa+cCl+cH+cOH+4*cCa+4*cMg+4*cCO3+cHCO3+cCaCl+cCaHCO3+cCaOH
                    +cMgCl+cMgHCO3+cMgOH+cNaCO3)
        if abs(Inew-I) < 1e-10: I = Inew; break
        I = 0.5*I + 0.5*Inew
    return dict(pH=pH, I=I, H=cH, OH=cOH, CO2=cCO2, HCO3=cHCO3, CO3=cCO3,
                Ca=cCa, CaCl=cCaCl, CaCO3=cCaCO3, CaHCO3=cCaHCO3, CaOH=cCaOH,
                Mg=cMg, MgCl=cMgCl, MgCO3=cMgCO3, MgHCO3=cMgHCO3, MgOH=cMgOH,
                Na=cNa, NaCl=cNaCl, NaCO3=cNaCO3, NaHCO3=cNaHCO3, NaOH=cNaOH, Cl=cCl)

## 4. pH from the mass balance (proton condition, fixed pCO₂)

With CO₂ as the fixed reference, the proton condition (total excess H⁺ relative to CO₂/H₂O and the
metal/Na components) is

$$P(\text{pH})=[H^+]-[OH^-]-[HCO_3^-]-2[CO_3^{2-}]-[NaHCO_3]-2[NaCO_3^-]-[CaHCO_3^+]-2[CaCO_3]-[CaOH^+]-[MgHCO_3^+]-2[MgCO_3]-[MgOH^+]-[NaOH]$$

MINTEQ sets the total H⁺ from the entered carbonate: the total CO₃²⁻ component `C_CO3` (the measured
CO₃²⁻) contributes `−2·C_CO3`. So pH is the root of

$$P(\text{pH})=-2\,C_{CO3}.$$

The free H⁺ starts from the neutral guess 10⁻⁷ (bracketed for the root find). Ca²⁺, Mg²⁺, Na⁺ and Cl⁻
carry no proton, so — unlike a charge balance — dissolved Ca/Mg do not drag the pH the wrong way.

In [4]:
def proton_condition(s):
    return (s['H'] - s['OH'] - s['HCO3'] - 2*s['CO3']
            - s['NaHCO3'] - 2*s['NaCO3'] - s['CaHCO3'] - 2*s['CaCO3'] - s['CaOH']
            - s['MgHCO3'] - 2*s['MgCO3'] - s['MgOH'] - s['NaOH'])

def solve_pH(CaT, MgT, NaT, ClT, C_CO3):
    f = lambda pH: proton_condition(speciate(pH, CaT, MgT, NaT, ClT)) + 2*C_CO3
    return brentq(f, 3.0, 11.5, xtol=1e-7)   # H+ from a neutral (1e-7) bracket

## 5. Measured data (run-aligned)

Columns: measured pH (reference only), Cl, Na, Ca, Mg, CO₃, HCO₃ (mg/L). `C_CO3` (the total CO₃²⁻
component MINTEQ uses to fix the pH) is the measured CO₃ column.

In [5]:
A = pd.DataFrame({'run':range(1,10),'pH':[8.9,8.8,8.9,9.0,9.0,8.8,8.8,9.0,8.9],
 'Cl':[48942,39648,42883,38612,33835,36206,32751,30515,32807],
 'Na':[27749.8,23604.3,34199.3,29672.0,24947.5,26775.3,24657.7,21705.0,23506.8],
 'Ca':[29.4,30.7,28.3,28.2,30.1,35.0,31.0,30.6,35.7],
 'Mg':[43.3,42.7,41.2,41.6,43.8,49.5,46.6,45.1,45.0],
 'CO3':[27.9,23.4,32.1,31.7,30.4,23.5,20.8,35.0,22.9],'HCO3':[119.8,127.8,107.8,116.1,109.7,125.5,131.5,121.0,128.3]})
B = pd.DataFrame({'run':range(1,10),'pH':[2.4,2.1,2.1,1.8,5.5,10.5,10.6,10.6,10.8],
 'Cl':[16937,17017,16827,16777,16838,16879,16854,16829,17022],
 'Na':[53900,32927,23180,24113,23718,21836,21302,21073,23484],
 'Ca':[0]*9,'Mg':[0]*9,'CO3':[0,0,0,0,0,48.0,46.6,55.0,80.1],'HCO3':[0]*9})
C = pd.DataFrame({'run':range(1,10),'pH':[7.5,7.5,7.2,7.0,8.8,9.8,10.2,10.2,10.6],
 'Cl':[24901,32258,36300,26800,24244,23699,21815,24118,22678],
 'Na':[19593.1,23296.4,25729.3,16953.4,16466.5,16183.2,14759.0,16450.8,15393.8],
 'Ca':[398.9,560.6,755.1,1325.3,37.2,15.6,16.3,17.0,9.8],
 'Mg':[114.2,145.9,180.4,233.9,24.4,13.6,4.4,3.6,1.0],
 'CO3':[0,0,0,0,18.2,92.8,120.2,134.0,136.4],'HCO3':[407.6,413.3,250.1,527.5,106.1,7.2,0,0,0]})

def totals(r):
    return (r['Ca']/MM['Ca']/1e3, r['Mg']/MM['Mg']/1e3, r['Na']/MM['Na']/1e3,
            r['Cl']/MM['Cl']/1e3, r['CO3']/MM['CO3']/1e3)   # Ca,Mg,Na,Cl, C_CO3

def run_all(df):
    out=[]
    for _,r in df.iterrows():
        CaT,MgT,NaT,ClT,Cc = totals(r)
        pH = solve_pH(CaT,MgT,NaT,ClT,Cc)
        s = speciate(pH,CaT,MgT,NaT,ClT); s['run']=int(r['run']); s['pH_meas']=r['pH']
        out.append(s)
    return out
resA, resB, resC = run_all(A), run_all(B), run_all(C)

## 6. Calculated pH vs Visual MINTEQ

Side by side with the MINTEQ output. The one row that differs, vessel B run 6, is because the MINTEQ
run for it carried no carbonate (its RUN5/RUN6 columns are identical); the model value is what a B run
with 48 mg/L CO₃ gives.

In [6]:
minteq = {'A':[8.03,7.957,8.052,8.054,8.045,7.942,7.894,8.104,7.936],
          'B':[5.476,5.536,5.56,5.558,5.559,5.559,8.196,8.259,8.387],
          'C':[5.561,5.551,5.543,5.556,7.85,8.482,8.579,8.611,8.622]}
for tag,res in [('A',resA),('B',resB),('C',resC)]:
    df = pd.DataFrame({'pH calc':[round(s['pH'],3) for s in res],
                       'pH MINTEQ':minteq[tag],
                       'I calc':[round(s['I'],3) for s in res]})
    df.index=[f'run {i+1}' for i in range(9)]; df.index.name=f'Vessel {tag}'
    print(df.to_string()); print()

          pH calc  pH MINTEQ  I calc
Vessel A                            
run 1       7.862      8.030   1.179
run 2       7.802      7.957   0.995
run 3       7.879      8.052   1.224
run 4       7.894      8.054   1.094
run 5       7.899      8.045   0.951
run 6       7.780      7.942   1.014
run 7       7.737      7.894   0.933
run 8       7.975      8.104   0.849
run 9       7.786      7.936   0.912

          pH calc  pH MINTEQ  I calc
Vessel B                            
run 1       5.405      5.476   1.331
run 2       5.475      5.536   0.907
run 3       5.511      5.560   0.707
run 4       5.508      5.558   0.726
run 5       5.509      5.559   0.718
run 6       8.097      5.559   0.681
run 7       8.088      8.196   0.670
run 8       8.159      8.259   0.665
run 9       8.301      8.387   0.716

          pH calc  pH MINTEQ  I calc
Vessel C                            
run 1       5.519      5.561   0.762
run 2       5.502      5.551   0.932
run 3       5.489      5.543   1.037

## 7. Full speciation at the calculated pH (mol/L) — compare with the MINTEQ sheet

In [7]:
order = ['H','OH','CO2','HCO3','CO3','Ca','CaCl','CaCO3','CaHCO3','CaOH',
         'Mg','MgCl','MgCO3','MgHCO3','MgOH','Na','NaCl','NaCO3','NaHCO3','NaOH','Cl']
def spec_table(res, tag):
    cols={f'run {s["run"]}':[s[k] for k in order] for s in res}
    df=pd.DataFrame(cols, index=order); df.index.name=f'Vessel {tag} (mol/L)'
    return df
for tag,res in [('A',resA),('B',resB),('C',resC)]:
    print(f'--- Vessel {tag} ---')
    print(spec_table(res,tag).to_string(float_format=lambda x: f'{x:.3e}')); print()

--- Vessel A ---
                     run 1     run 2     run 3     run 4     run 5     run 6     run 7     run 8     run 9
Vessel A (mol/L)                                                                                          
H                1.672e-08 1.998e-08 1.590e-08 1.582e-08 1.612e-08 2.091e-08 2.346e-08 1.379e-08 2.108e-08
OH               8.942e-07 8.106e-07 9.210e-07 9.821e-07 1.023e-06 7.687e-07 7.080e-07 1.243e-06 7.942e-07
CO2              9.595e-06 1.001e-05 9.496e-06 9.785e-06 1.011e-05 9.969e-06 1.016e-05 1.035e-05 1.021e-05
HCO3             5.028e-04 4.558e-04 5.179e-04 5.523e-04 5.751e-04 4.323e-04 3.981e-04 6.988e-04 4.466e-04
CO3              3.081e-06 2.742e-06 3.201e-06 3.860e-06 4.444e-06 2.447e-06 2.145e-06 6.816e-06 2.720e-06
Ca               6.551e-04 7.066e-04 6.374e-04 6.466e-04 7.017e-04 8.107e-04 7.257e-04 7.199e-04 8.360e-04
CaCl             7.614e-05 5.754e-05 6.618e-05 5.462e-05 4.684e-05 6.055e-05 4.626e-05 4.057e-05 5.273e-05
CaCO3            6.8

## 8. Net surface charge — Pokrovsky Eqn 1

At the calculated pH, $\sigma_T=\big(\tfrac12(q_A+q_B)-q_C\big)/S$ with $q=\sum_k z_k[k]$ over the
reactive species (Na⁺, Cl⁻ cancel in the difference).

In [8]:
def qcharge(s):
    q_H   = s['H'] - s['OH']
    q_DIC = -s['HCO3'] - 2*s['CO3'] - s['NaCO3']
    q_Ca  = 2*s['Ca'] + s['CaCl'] + s['CaHCO3'] + s['CaOH']
    q_Mg  = 2*s['Mg'] + s['MgCl'] + s['MgHCO3'] + s['MgOH']
    return q_H + q_DIC + q_Ca + q_Mg     # (CaCl+, MgCl+ are +1; Na+, Cl- omitted)

rows=[]
for i in range(9):
    q0 = 0.5*(qcharge(resA[i]) + qcharge(resB[i]))
    qf = qcharge(resC[i])
    rows.append({'run':i+1,'pH_C':round(resC[i]['pH'],3),'sigma_T (mmol/m2)':(q0-qf)/St*1e3})
print(pd.DataFrame(rows).set_index('run').to_string(float_format=lambda x: f'{x:.4g}'))

     pH_C  sigma_T (mmol/m2)
run                         
1   5.519             -1.043
2   5.502             -1.428
3   5.489             -1.893
4   5.516             -3.196
5   7.726           -0.04869
6   8.413            0.09053
7   8.527             0.1398
8   8.561             0.1386
9   8.574             0.1653


## Notes

1. **Fixed pCO₂ + pH from mass balance**, matching Visual MINTEQ option 1: {CO₂(aq)} is fixed at
   10⁻⁴·⁹, and pH is the root of the proton condition `P(pH) = −2·C_CO3`, with H⁺ started from 10⁻⁷.
2. **Ca²⁺, Mg²⁺, Na⁺, Cl⁻ carry no proton**, so dissolved Ca/Mg no longer push the pH the wrong way —
   the calculated pH of vessel C now falls with acid and rises with base, as MINTEQ gives.
3. **Davies activity coefficients** (with the 0.1·I Setchenow term for neutrals) and the
   TOUGHREACT/EQ3-6 (Plummer–Busenberg) constants, as in MINTEQ. The residual ~0.1 pH offset is the
   activity-model detail (MINTEQ's ion pairing gives a slightly lower ionic strength).
4. **No surface complexation model.** The surface charge is the aqueous charge-sum difference of
   Pokrovsky Eqn 1 at the calculated pH.